In [251]:
import os
import pandas as pd
from pathlib import Path
import gc

# Run garbage collection
gc.collect()

data_folder = Path(r"/kaggle/input/competitions/name-that-llm/train.csv")

data_path = os.path.join(os.getcwd(), data_folder)


print(data_path)

df = pd.read_csv(data_path)

df.head()

/kaggle/input/competitions/name-that-llm/train.csv


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash
3,009319596f10,p073,Data science communities matter because they t...,gpt-5.6-terra
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",glm-5.3-flash


In [252]:
import torch
torch.cuda.device_count()

2

In [253]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["model"] = le.fit_transform(df["model"])
le.transform(le.classes_)
df.head()

,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",17
1,0041ac86ee93,p073,Data science communities matter because they a...,4
2,005a19083eae,p120,"In the final twilight of time, the last embers...",6
3,009319596f10,p073,Data science communities matter because they t...,14
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",10


In [254]:
!pip install psycopg

In [255]:
from psycopg.generators import fetch
import re
import numpy as np
import pandas as pd
import nltk
nltk.download("punkt")
from nltk.tokenize import sent_tokenize, word_tokenize

features = pd.DataFrame(index=df.index)
features["model"] = df["model"]
features["text"] = df["text"]
text_col = "text"

# Character, word, and sentence counts
features["char_count"] = df[text_col].apply(len)
# Words & Sentences
def get_words_and_sents(text):
        sents = sent_tokenize(text) if len(text.strip()) > 0 else [""]
        words = word_tokenize(text) if len(text.strip()) > 0 else [""]
        return len(words), len(sents), sents, words

tokenized_info = df[text_col].apply(get_words_and_sents)
features["word_count"] = tokenized_info.apply(lambda x: x[0])
features["sent_count"] = tokenized_info.apply(lambda x: x[1])
# Sentence Length Mean & Variance
def sent_stats(info):
    _, num_sents, sents, _ = info
    if num_sents == 0:
        return 0.0, 0.0
    lengths = [len(s.split()) for s in sents]
    return float(np.mean(lengths)), float(np.var(lengths))

sent_metrics = tokenized_info.apply(sent_stats)
features["avg_sent_len"] = sent_metrics.apply(lambda x: x[0])
features["var_sent_len"] = sent_metrics.apply(lambda x: x[1])
# Avg Word Length
features["avg_word_len"] = features["char_count"] / (features["word_count"] + 1e-5)
# ----------------- 2. Markdown Stylometry -----------------
features["md_asterisk_count"] = df[text_col].apply(lambda x: x.count("*"))  # Bold or italics (* or **)
# ----------------- 3. Punctuation Fingerprint -----------------
features["punct_colon"] = df[text_col].apply(lambda x: x.count(":"))
features["punct_semicolon"] = df[text_col].apply(lambda x: x.count(";"))
features["punct_em_dash"] = df[text_col].apply(lambda x: x.count("—") + x.count("--"))
features["punct_exclamation"] = df[text_col].apply(lambda x: x.count("!"))
features["punct_question"] = df[text_col].apply(lambda x: x.count("?"))
features["punct_comma"] = df[text_col].apply(lambda x: x.count(","))
features["punct_period"] = df[text_col].apply(lambda x: x.count("."))
features["punct_quotes"] = df[text_col].apply(lambda x: x.count('"') + x.count("'"))
features["punct_parens"] = df[text_col].apply(lambda x: x.count("(") + x.count(")"))
features["punct_brackets"] = df[text_col].apply(lambda x: x.count("[") + x.count("]"))
# Punctuation Density (normalized by word count)
total_puncts = (features["punct_colon"] + features["punct_semicolon"] + 
                features["punct_comma"] + features["punct_period"] + features["punct_exclamation"])
features["punct_density"] = total_puncts / (features["word_count"] + 1e-5)
# ----------------- 4. Capitalization & Casing -----------------
features["upper_char_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isupper()))
features["upper_char_ratio"] = features["upper_char_count"] / (features["char_count"] + 1e-5)
features["all_caps_words"] = df[text_col].apply(lambda x: len(re.findall(r"\b[A-Z]{2,}\b", x)))
features["camel_case_count"] = df[text_col].apply(lambda x: len(re.findall(r"\b[a-z]+[A-Z][a-zA-Z]*\b", x)))
# ----------------- 5. Lexical Diversity (Type-Token Ratio) -----------------
def type_token_ratio(info):
    num_words, _, _, words = info
    if num_words == 0:
        return 0.0
    unique_words = set(w.lower() for w in words if w.isalnum())
    return len(unique_words) / (num_words + 1e-5)

features["lexical_diversity_ttr"] = tokenized_info.apply(type_token_ratio)
# Digits / Numbers presence
features["digit_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isdigit()))
features["digit_ratio"] = features["digit_count"] / (features["char_count"] + 1e-5)
features.head()

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


,model,text,char_count,word_count,sent_count,avg_sent_len,var_sent_len,avg_word_len,md_asterisk_count,punct_colon,...,punct_parens,punct_brackets,punct_density,upper_char_count,upper_char_ratio,all_caps_words,camel_case_count,lexical_diversity_ttr,digit_count,digit_ratio
0,17,"When a language dies, a particular astronomy d...",1143,222,8,24.000000,91.000000,5.148648,0,1,...,0,0,0.126126,10,0.008749,0,0,0.617117,0,0.0
1,4,Data science communities matter because they a...,518,74,4,16.250000,10.687500,6.999999,0,0,...,0,0,0.121622,4,0.007722,0,0,0.783784,0,0.0
2,6,"In the final twilight of time, the last embers...",474,89,4,19.250000,17.687500,5.325842,0,0,...,0,0,0.134831,4,0.008439,0,0,0.707865,0,0.0
3,14,Data science communities matter because they t...,1724,263,9,24.444444,17.135802,6.555133,0,1,...,0,0,0.163498,9,0.005220,0,0,0.669201,0,0.0
4,10,"Deep inside each ear, three tiny loops of bone...",541,110,4,24.250000,8.687500,4.918181,0,1,...,0,0,0.118182,4,0.007394,0,0,0.690909,0,0.0


### DeBERTa -> Tokenizer 

In [256]:
# import os
# HF_TOKEN = os.getenv("HF_TOKEN")
# print(HF_TOKEN)

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
# print(HF_TOKEN)


In [257]:
from transformers import ( AutoModelForSequenceClassification, AutoTokenizer)

# LOCAL_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\checkpoints"
NUM_LABELS = 18
MODEL_NAME = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, tokens=HF_TOKEN)

model = AutoModelForSequenceClassification.from_pretrained(
       MODEL_NAME, 
       num_labels=NUM_LABELS,
     torch_dtype=torch.float32
    )

# model.save_pretrained(LOCAL_DIR)
# tokenizer.save_pretrained(LOCAL_DIR)
model


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifie

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

In [258]:
# #  switch off the the trainable parameters
# for param in model.deberta.parameters():
#     param.requires_grad = False

In [259]:
# Total and trainable parameters
total_params = model.num_parameters()
trainable_params = model.num_parameters(only_trainable=True)
print(f"Total: {total_params:,} | Trainable: {trainable_params:,}")

Total: 141,908,754 | Trainable: 141,908,754


In [260]:

# Total parameters (including frozen layers)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")


Total Parameters: 141,908,754


In [261]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

141908754

In [262]:
# Trainable parameters only (layers being updated during training)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable Parameters: {trainable_params:,}")

Trainable Parameters: 141,908,754


In [263]:
# # Layer wise parameter
# for param in model.parameters():
#     print(param.shape)
# for name, param in model.named_parameters():
#      print(f"Layer: {name} | Shape: {param.shape}")


In [264]:
text = [
    "The proposed method is efficient.",
    "This answer provides a detailed explanation.",
    "The model generates concise responses."
]

# 	******"padding=True" ka matlab yahan batch ke longest sequence tak padding karna hai.
#    Agar padding="max_length" use karte, toh har sequence ko specified max_length tak pad kiya jata.*****
encoded = tokenizer(
    text,
    truncation=True,
    max_length=128,
    padding=True,
    return_tensors="pt"
)

print(encoded["input_ids"].shape)
print(encoded["attention_mask"].shape)

torch.Size([3, 9])
torch.Size([3, 9])


In [265]:
from torch.onnx._internal.torchscript_exporter.symbolic_opset9 import dim
import torch
outputs = model(**encoded)
logits = outputs.logits
pred = torch.argmax(logits, dim=1)
pred

prob = torch.softmax(logits, dim=1)
prob

predictions = torch.argmax(prob, dim=1)
predictions

tensor([12, 12, 12])

In [266]:
import torch
from torch.utils.data import Dataset

class LLMTextData(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length
        if len(self.texts) != len(self.labels):
            raise ValueError("Texts and labels are not of equal length")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoded = self.tokenizer(
            text, 
            truncation=True,
            padding="max_length",
            max_length = self.max_length,
            return_tensors = "pt"
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),     # (1, 10) -> (10,)
            "attention_mask": encoded["attention_mask"].squeeze(0),   # (1, 10) -> (10,)
            "labels": torch.tensor(label, dtype=torch.long)
        }

        return item

In [267]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(features, test_size=0.20, random_state=42, stratify=df["model"])

print(train_df.shape, test_df.shape)

train_datasets = LLMTextData(
    texts=train_df["text"],
    labels=train_df["model"],
    max_length=256,
    tokenizer=tokenizer
)

test_datasets = LLMTextData(
    texts=test_df["text"],
    labels=test_df["model"],
    max_length=256,
    tokenizer=tokenizer
)

# Dynamic padding
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)
# DataLoader
train_loader = DataLoader(
    train_datasets,
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)

# train_loader = DataLoader(
#     dataset=train_datasets,
#     batch_size=8,
#     shuffle=True,
# )

test_loader = DataLoader(
    dataset=test_datasets,
    batch_size=8,
)

print(len(train_loader), len(test_loader))

(2159, 27) (540, 27)
270 68


In [268]:
# test_df["model"]

In [269]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)

cuda


In [270]:
# import torch.nn as nn
# from torch.optim import AdamW

# model = model.to(device)

# if torch.cuda.device_count() > 1:
#     model = nn.DataParallel(model)

# criterion = nn.CrossEntropyLoss()
# optimizer = AdamW(
#     model.parameters(),
#     lr=1e-5,
#     weight_decay=0.01
# )

import torch.nn as nn
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup  # or get_linear_schedule_with_warmup

# --- Option A: Unfreeze whole backbone for best performance (Recommended) ---
for param in model.deberta.parameters():
    param.requires_grad = True

# --- Option B: If GPU memory is tight, unfreeze only the last 2 layers ---
# for param in model.deberta.encoder.layer[-2:].parameters():
#     param.requires_grad = True


# 2. Move to device & wrap with DataParallel if multi-GPU
model = model.to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

# 3. Access base model safely (works with or without DataParallel)
base_model = getattr(model, "module", model)

criterion = nn.CrossEntropyLoss()

EPOCHS = 15
# Check if backbone is trainable using base_model
LR = 2e-5 if any(p.requires_grad for p in base_model.deberta.parameters()) else 5e-4

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)




In [271]:

# from tqdm.auto import tqdm

# EPOCHS = 15

# for epoch in range(EPOCHS):
#     model.train()
#     total_loss = 0.0
#     total_examples = 0
#     correct_predictions = 0

#     progress_bar = tqdm(
#         train_loader,
#         desc=f"Epoch: {epoch + 1}/{EPOCHS}"
#     )

#     for batch in progress_bar:
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#         labels = batch["labels"].to(device)

#         optimizer.zero_grad()

#         # Forward pass
#         outputs = model(
#             input_ids=input_ids,
#             attention_mask=attention_mask
#         )

#         logits = outputs.logits
#         # print(logits)
#         loss = criterion(logits, labels)

#         # Backpropagation and weight update
#         loss.backward()
#         optimizer.step()

#         # Metrics
#         batch_size = labels.size(0)
#         predictions = torch.argmax(logits, dim=1)

#         correct_predictions += (predictions == labels).sum().item()
#         total_loss += loss.item() * batch_size
#         total_examples += batch_size

#         # Update progress bar after every batch
#         progress_bar.set_postfix(
#             loss=f"{total_loss / total_examples:.4f}",
#             accuracy=f"{correct_predictions / total_examples:.4f}"
#         )

#     epoch_loss = total_loss / total_examples
#     epoch_acc = correct_predictions / total_examples

#     print(
#         f"Epoch {epoch + 1}/{EPOCHS} | "
#         f"Loss: {epoch_loss:.4f} | "
#         f"Accuracy: {epoch_acc:.4f}"
#     )

from tqdm.auto import tqdm

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    total_examples = 0
    correct_predictions = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch: {epoch + 1}/{EPOCHS}"
    )

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        loss = criterion(logits, labels)

        # Backpropagation and weight update
        loss.backward()
        
        # Recommended: gradient clipping to avoid gradient explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()  # <-- STEP SCHEDULER HERE ON EACH BATCH

        # Metrics
        batch_size = labels.size(0)
        predictions = torch.argmax(logits, dim=1)

        correct_predictions += (predictions == labels).sum().item()
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        # Current learning rate for logging
        current_lr = scheduler.get_last_lr()[0]

        # Update progress bar
        progress_bar.set_postfix(
            loss=f"{total_loss / total_examples:.4f}",
            acc=f"{correct_predictions / total_examples:.4f}",
            lr=f"{current_lr:.2e}"
        )

    epoch_loss = total_loss / total_examples
    epoch_acc = correct_predictions / total_examples

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_acc:.4f}"
    )



Epoch: 1/15:   0%|          | 0/270 [00:00<?, ?it/s]

Epoch 1/15 | Loss: 2.9087 | Accuracy: 0.0584


Epoch: 2/15:   0%|          | 0/270 [00:00<?, ?it/s]

Epoch 2/15 | Loss: 2.4969 | Accuracy: 0.1366


Epoch: 3/15:   0%|          | 0/270 [00:00<?, ?it/s]

...

Epoch 13/15 | Loss: 0.4145 | Accuracy: 0.8884


Epoch: 14/15:   0%|          | 0/270 [00:00<?, ?it/s]

Epoch 14/15 | Loss: 0.3733 | Accuracy: 0.9078


Epoch: 15/15:   0%|          | 0/270 [00:00<?, ?it/s]

Epoch 15/15 | Loss: 0.3658 | Accuracy: 0.9134


In [272]:
from sklearn.metrics import accuracy_score, recall_score
from tqdm.auto import tqdm

y_true = []
y_pred = []
total_test_batches = len(test_loader)


model.eval()
with torch.no_grad():
    progress_bar = tqdm(test_loader, desc=f"Predicting....")
    for cnt, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1)

        y_true.extend(labels.tolist())
        y_pred.extend(pred.cpu().tolist())
        # print(f"Predicting: {cnt+1}/{total_test_batches}")


print("Accuracy:", accuracy_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred, average="macro"))
        


        


Predicting....:   0%|          | 0/68 [00:00<?, ?it/s]

Accuracy: 0.3277777777777778
Recall: 0.3277777777777778


In [273]:
# from psycopg.generators import fetch
# import re
# import numpy as np
# import pandas as pd
# import nltk
# nltk.download("punkt")
# from nltk.tokenize import sent_tokenize, word_tokenize

# features = pd.DataFrame(index=df.index)
# features["model"] = df["model"]
# text_col = "text"

In [274]:
# # Character, word, and sentence counts
# features["char_count"] = df[text_col].apply(len)
# # Words & Sentences
# def get_words_and_sents(text):
#         sents = sent_tokenize(text) if len(text.strip()) > 0 else [""]
#         words = word_tokenize(text) if len(text.strip()) > 0 else [""]
#         return len(words), len(sents), sents, words

# tokenized_info = df[text_col].apply(get_words_and_sents)
# features["word_count"] = tokenized_info.apply(lambda x: x[0])
# features["sent_count"] = tokenized_info.apply(lambda x: x[1])
# # Sentence Length Mean & Variance
# def sent_stats(info):
#     _, num_sents, sents, _ = info
#     if num_sents == 0:
#         return 0.0, 0.0
#     lengths = [len(s.split()) for s in sents]
#     return float(np.mean(lengths)), float(np.var(lengths))

# sent_metrics = tokenized_info.apply(sent_stats)
# features["avg_sent_len"] = sent_metrics.apply(lambda x: x[0])
# features["var_sent_len"] = sent_metrics.apply(lambda x: x[1])
# # Avg Word Length
# features["avg_word_len"] = features["char_count"] / (features["word_count"] + 1e-5)
# # ----------------- 2. Markdown Stylometry -----------------
# features["md_asterisk_count"] = df[text_col].apply(lambda x: x.count("*"))  # Bold or italics (* or **)
# # ----------------- 3. Punctuation Fingerprint -----------------
# features["punct_colon"] = df[text_col].apply(lambda x: x.count(":"))
# features["punct_semicolon"] = df[text_col].apply(lambda x: x.count(";"))
# features["punct_em_dash"] = df[text_col].apply(lambda x: x.count("—") + x.count("--"))
# features["punct_exclamation"] = df[text_col].apply(lambda x: x.count("!"))
# features["punct_question"] = df[text_col].apply(lambda x: x.count("?"))
# features["punct_comma"] = df[text_col].apply(lambda x: x.count(","))
# features["punct_period"] = df[text_col].apply(lambda x: x.count("."))
# features["punct_quotes"] = df[text_col].apply(lambda x: x.count('"') + x.count("'"))
# features["punct_parens"] = df[text_col].apply(lambda x: x.count("(") + x.count(")"))
# features["punct_brackets"] = df[text_col].apply(lambda x: x.count("[") + x.count("]"))
# # Punctuation Density (normalized by word count)
# total_puncts = (features["punct_colon"] + features["punct_semicolon"] + 
#                 features["punct_comma"] + features["punct_period"] + features["punct_exclamation"])
# features["punct_density"] = total_puncts / (features["word_count"] + 1e-5)
# # ----------------- 4. Capitalization & Casing -----------------
# features["upper_char_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isupper()))
# features["upper_char_ratio"] = features["upper_char_count"] / (features["char_count"] + 1e-5)
# features["all_caps_words"] = df[text_col].apply(lambda x: len(re.findall(r"\b[A-Z]{2,}\b", x)))
# features["camel_case_count"] = df[text_col].apply(lambda x: len(re.findall(r"\b[a-z]+[A-Z][a-zA-Z]*\b", x)))
# # ----------------- 5. Lexical Diversity (Type-Token Ratio) -----------------
# def type_token_ratio(info):
#     num_words, _, _, words = info
#     if num_words == 0:
#         return 0.0
#     unique_words = set(w.lower() for w in words if w.isalnum())
#     return len(unique_words) / (num_words + 1e-5)

# features["lexical_diversity_ttr"] = tokenized_info.apply(type_token_ratio)
# # Digits / Numbers presence
# features["digit_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isdigit()))
# features["digit_ratio"] = features["digit_count"] / (features["char_count"] + 1e-5)
# features.head()

In [275]:
train_df

,model,text,char_count,word_count,sent_count,avg_sent_len,var_sent_len,avg_word_len,md_asterisk_count,punct_colon,...,punct_parens,punct_brackets,punct_density,upper_char_count,upper_char_ratio,all_caps_words,camel_case_count,lexical_diversity_ttr,digit_count,digit_ratio
231,12,Algorithmic echo chambers do not merely person...,501,74,3,21.666667,29.555556,6.770269,0,1,...,0,0,0.121622,3,0.005988,0,0,0.851351,0,0.00000
1034,1,The bystander effect reveals a paradox of soci...,371,60,2,26.500000,6.250000,6.183332,0,1,...,0,0,0.100000,2,0.005391,0,0,0.766667,0,0.00000
1225,6,For a lifetime bound within the sterile geomet...,1278,230,7,28.714286,103.346939,5.556521,0,0,...,0,0,0.126087,9,0.007042,0,0,0.643478,0,0.00000
891,4,The uncanny valley opens not as a gradual slop...,1150,196,5,35.000000,52.000000,5.867347,0,2,...,0,0,0.107143,5,0.004348,0,0,0.683673,0,0.00000
1956,11,To build and dwell upon a stellar Dyson sphere...,267,49,1,43.000000,0.000000,5.448978,0,0,...,0,0,0.081633,2,0.007491,0,0,0.714286,0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276,8,"Beneath the forest floor lies an ancient, puls...",406,65,3,19.000000,32.666667,6.246153,0,0,...,0,0,0.123077,3,0.007389,0,0,0.861538,0,0.00000
106,16,"Deep within the entorhinal cortex, grid cells ...",554,94,3,27.000000,88.666667,5.893616,0,1,...,0,0,0.127660,9,0.016245,0,0,0.744681,4,0.00722
76,8,Consider the walking mirror: an entity that la...,589,103,4,21.750000,60.687500,5.718446,0,1,...,0,0,0.145631,5,0.008489,0,0,0.737864,0,0.00000
493,9,The inverted spectrum problem exposes a chilli...,258,48,2,21.500000,6.250000,5.374999,0,1,...,0,0,0.104167,2,0.007752,0,0,0.854166,0,0.00000


In [276]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

#  Train-Test Split (stratified)
# df_train, df_test = train_test_split(features, test_size=0.2, random_state=42, stratify=features["model"])
y_train = train_df["model"].values
y_test = test_df["model"].values
X_train_pre = train_df.iloc[:, 2:]
X_test_pre = test_df.iloc[:, 2:]


X_train_pre.head()

,char_count,word_count,sent_count,avg_sent_len,var_sent_len,avg_word_len,md_asterisk_count,punct_colon,punct_semicolon,punct_em_dash,...,punct_parens,punct_brackets,punct_density,upper_char_count,upper_char_ratio,all_caps_words,camel_case_count,lexical_diversity_ttr,digit_count,digit_ratio
231,501,74,3,21.666667,29.555556,6.770269,0,1,1,0,...,0,0,0.121622,3,0.005988,0,0,0.851351,0,0.0
1034,371,60,2,26.500000,6.250000,6.183332,0,1,0,0,...,0,0,0.100000,2,0.005391,0,0,0.766667,0,0.0
1225,1278,230,7,28.714286,103.346939,5.556521,0,0,1,0,...,0,0,0.126087,9,0.007042,0,0,0.643478,0,0.0
891,1150,196,5,35.000000,52.000000,5.867347,0,2,0,2,...,0,0,0.107143,5,0.004348,0,0,0.683673,0,0.0
1956,267,49,1,43.000000,0.000000,5.448978,0,0,0,0,...,0,0,0.081633,2,0.007491,0,0,0.714286,0,0.0


In [277]:

X_train_pre.shape, X_test_pre.shape
# X_train.head()
encoder = StandardScaler()
X_train = encoder.fit_transform(X_train_pre)
X_test = encoder.transform(X_test_pre)
X_train.shape, X_test.shape
X_train

array([[-0.67748689, -0.88909856, -0.86184926, ...,  2.18089009,
        -0.27476085, -0.27539568],
       [-1.02243627, -1.10107958, -1.32460294, ...,  0.96690024,
        -0.27476085, -0.27539568],
       [ 1.38424906,  1.4729756 ,  0.98916546, ..., -0.79905461,
        -0.27476085, -0.27539568],
       ...,
       [-0.44398269, -0.44999503, -0.39909558, ...,  0.55400413,
        -0.27476085, -0.27539568],
       [-1.32227689, -1.28277759, -1.32460294, ...,  2.22124789,
        -0.27476085, -0.27539568],
       [-1.08346578, -0.99508907, -1.32460294, ...,  1.31637057,
        -0.27476085, -0.27539568]])

In [278]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
clf = LogisticRegression(max_iter=1000, C=2.0, verbose=1)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall Score: {recall_score(y_test, y_pred, average='macro'):.4f}")
# print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")



Accuracy Score: 0.2611
Recall Score: 0.2611


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.2s finished


In [279]:
import torch
import numpy as np

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

# ==========================================
# 1. DeBERTa Predictions
# ==========================================

model.eval()

deberta_probs = []
y_true = []

with torch.no_grad():
    progress_bar = tqdm(test_loader, desc="DeBERTa Predictions")

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Convert logits into probabilities
        probs = torch.softmax(outputs.logits, dim=1)

        deberta_probs.extend(probs.cpu().numpy())
        y_true.extend(labels.cpu().numpy())


deberta_probs = np.array(deberta_probs)
y_true = np.array(y_true)

print("DeBERTa probability shape:", deberta_probs.shape)



DeBERTa Predictions:   0%|          | 0/68 [00:00<?, ?it/s]

DeBERTa probability shape: (540, 18)


In [280]:

# ==========================================
# 2. Logistic Regression Probabilities
# ==========================================

# X_test must contain the same test samples
# in exactly the same order as test_loader.

lr_probs = clf.predict_proba(X_test)

print("Logistic Regression probability shape:", lr_probs.shape)

Logistic Regression probability shape: (540, 18)


In [281]:
# ==========================================
# 3. Soft Voting Ensemble
# ==========================================

# Equal weights: 50% DeBERTa + 50% Logistic Regression

ensemble_probs = (
    0.5 * deberta_probs +
    0.5 * lr_probs
)

ensemble_preds = np.argmax(ensemble_probs, axis=1)


# ==========================================
# 4. Evaluate Ensemble
# ==========================================

print("\n========== ENSEMBLE RESULTS ==========")

print(
    "Accuracy:",
    accuracy_score(y_true, ensemble_preds)
)

print(
    "Macro Recall:",
    recall_score(
        y_true,
        ensemble_preds,
        average="macro"
    )
)

print(
    "Confusion Matrix:\n",
    confusion_matrix(y_true, ensemble_preds)
)


========== ENSEMBLE RESULTS ==========
Accuracy: 0.34074074074074073
Macro Recall: 0.3407407407407408
Confusion Matrix:
 [[ 3  1 11  1  3  0  0  0  0  0  9  0  1  0  0  1  0  0]
 [ 0 15 12  0  1  0  0  0  0  1  1  0  0  0  0  0  0  0]
 [ 1  3 16  1  5  0  0  0  0  0  1  0  1  0  0  0  0  2]
 [ 2  0  0 16  1  0  0  0  0  5  5  0  0  0  0  1  0  0]
 [ 2  6 11  0 10  0  0  0  0  1  0  0  0  0  0  0  0  0]
 [ 0  1  0  0  0 23  6  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  1  0  0 12 15  1  0  0  0  0  0  0  0  0  0  1]
 [ 0  0  0  0  0  6 13  8  2  0  0  0  0  0  0  0  0  1]
 [ 0  0  1  0  0  3 20  5  1  0  0  0  0  0  0  0  0  0]
 [ 9  2  9  1  1  0  0  0  0  2  3  0  1  1  0  0  0  1]
 [ 3  1  6  2  2  0  0  0  0  5  8  0  0  0  0  2  1  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  5  8  7  6  2  1  1]
 [ 0  1  1  0  0  0  0  0  0  0  0  0 15  1  1  8  0  3]
 [ 0  0  1  0  0  0  0  0  0  0  0  0 13  4  4  6  0  2]
 [ 0  0  1  0  0  0  0  0  0  0  0  0 20  1  4  3  0  1]
 [ 0  0  1  0  0  0  0 

In [282]:
weights = [
    (0.5, 0.5),
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2),
    (0.9, 0.1)
]

for w_deberta, w_lr in weights:

    probs = (
        w_deberta * deberta_probs +
        w_lr * lr_probs
    )

    preds = np.argmax(probs, axis=1)

    acc = accuracy_score(y_true, preds)

    recall = recall_score(
        y_true,
        preds,
        average="macro"
    )

    print(
        f"DeBERTa={w_deberta:.1f}, "
        f"LR={w_lr:.1f} | "
        f"Accuracy={acc:.4f} | "
        f"Macro Recall={recall:.4f}"
    )

DeBERTa=0.5, LR=0.5 | Accuracy=0.3407 | Macro Recall=0.3407
DeBERTa=0.6, LR=0.4 | Accuracy=0.3426 | Macro Recall=0.3426
DeBERTa=0.7, LR=0.3 | Accuracy=0.3407 | Macro Recall=0.3407
DeBERTa=0.8, LR=0.2 | Accuracy=0.3315 | Macro Recall=0.3315
DeBERTa=0.9, LR=0.1 | Accuracy=0.3278 | Macro Recall=0.3278


In [310]:
from sklearn.metrics import balanced_accuracy_score, accuracy_score, recall_score
import numpy as np

# y_true: actual labels
# ensemble_preds: final ensemble predictions

balanced_acc = balanced_accuracy_score(y_true, ensemble_preds)
macro_recall = recall_score(y_true, ensemble_preds, average="macro")
accuracy = accuracy_score(y_true, ensemble_preds)

print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Macro Recall:      {macro_recall:.4f}")
print(f"Accuracy:          {accuracy:.4f}")

# Both metrics should be equal
assert np.isclose(balanced_acc, macro_recall)

Balanced Accuracy: 0.3407
Macro Recall:      0.3407
Accuracy:          0.3407


In [311]:
from sklearn.metrics import balanced_accuracy_score
import numpy as np

# Individual model predictions
deberta_preds = np.argmax(deberta_probs, axis=1)
lr_preds = np.argmax(lr_probs, axis=1)
ensemble_preds = np.argmax(ensemble_probs, axis=1)

# Compare performance
print(f"DeBERTa Balanced Accuracy:   {balanced_accuracy_score(y_true, deberta_preds):.4f}")
print(f"Logistic Regression:         {balanced_accuracy_score(y_true, lr_preds):.4f}")
print(f"Ensemble Balanced Accuracy:  {balanced_accuracy_score(y_true, ensemble_preds):.4f}")

DeBERTa Balanced Accuracy:   0.3278
Logistic Regression:         0.2611
Ensemble Balanced Accuracy:  0.3407


In [283]:
import os
import torch

SAVE_DIR = "ensemble_pipeline_2"

DEBERTA_DIR = os.path.join(SAVE_DIR, "deberta")
os.makedirs(DEBERTA_DIR, exist_ok=True)

# Save trained model weights and configuration
model_to_save = (model.module if isinstance(model, torch.nn.DataParallel) else model)
model_to_save.save_pretrained(DEBERTA_DIR)

# Save tokenizer
tokenizer.save_pretrained(DEBERTA_DIR)

print("DeBERTa model and tokenizer saved!")




Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DeBERTa model and tokenizer saved!


In [284]:
import os
import joblib

LR_DIR = os.path.join(SAVE_DIR, "logistic_regression")
os.makedirs(LR_DIR, exist_ok=True)

joblib.dump(clf, os.path.join(LR_DIR, "classifier.joblib"))

print("Logistic Regression model saved!")

Logistic Regression model saved!


In [306]:
import os
import pickle 

# Ensure os is imported to use os.path.join
path_ = os.path.join(LR_DIR, "encoder.pkl")

with open(path_, "wb") as f:
    pickle.dump(encoder, f)  


In [285]:
ensemble_config = {
    "deberta_weight": 0.5,
    "lr_weight": 0.5,
    "max_length": 256
}

joblib.dump(
    ensemble_config,
    os.path.join(SAVE_DIR, "ensemble_config.joblib")
)

print("Ensemble configuration saved!")

Ensemble configuration saved!


In [286]:
import torch
import joblib

from transformers import AutoTokenizer, AutoModelForSequenceClassification

SAVE_DIR = "ensemble_pipeline_2"

# Load DeBERTa
tokenizer = AutoTokenizer.from_pretrained(
    os.path.join(SAVE_DIR, "deberta")
)

model = AutoModelForSequenceClassification.from_pretrained(
    os.path.join(SAVE_DIR, "deberta")
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()


# Load Logistic Regression
LR_DIR = os.path.join(SAVE_DIR, "logistic_regression")

clf = joblib.load(
    os.path.join(LR_DIR, "classifier.joblib")
)

# Load ensemble configuration
config = joblib.load(
    os.path.join(SAVE_DIR, "ensemble_config.joblib")
)

print("Both models loaded successfully!")

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Both models loaded successfully!


In [287]:
# import numpy as np
# import torch

# def predict_ensemble(texts):

#     # -------------------------------
#     # 1. DeBERTa Predictions
#     # -------------------------------

#     model.eval()

#     inputs = tokenizer(
#         texts,
#         padding=True,
#         truncation=True,
#         max_length=config["max_length"],
#         return_tensors="pt"
#     )

#     inputs = {
#         key: value.to(device)
#         for key, value in inputs.items()
#     }

#     with torch.no_grad():

#         outputs = model(**inputs)

#         deberta_probs = torch.softmax(
#             outputs.logits,
#             dim=1
#         ).cpu().numpy()


#     # -------------------------------
#     # 2. Logistic Regression
#     # -------------------------------

#     X_text = vectorizer.transform(texts)

#     lr_probs = clf.predict_proba(X_text)


#     # -------------------------------
#     # 3. Align Class Probabilities
#     # -------------------------------

#     # Assumption: DeBERTa class indices correspond to
#     # labels 0, 1, ..., num_classes - 1.

#     num_classes = deberta_probs.shape[1]

#     lr_aligned = np.zeros_like(deberta_probs)

#     for j, label in enumerate(clf.classes_):

#         label_index = int(label)

#         if 0 <= label_index < num_classes:
#             lr_aligned[:, label_index] = lr_probs[:, j]
#         else:
#             raise ValueError(
#                 f"Unexpected Logistic Regression class: {label}"
#             )


#     # -------------------------------
#     # 4. Weighted Soft Voting
#     # -------------------------------

#     ensemble_probs = (
#         config["deberta_weight"] * deberta_probs
#         + config["lr_weight"] * lr_aligned
#     )

#     predictions = np.argmax(
#         ensemble_probs,
#         axis=1
#     )

#     confidence = np.max(
#         ensemble_probs,
#         axis=1
#     )

#     return predictions, confidence, ensemble_probs

In [309]:
# import shutil

# shutil.make_archive(
#     "ensemble_pipeline_1",
#     "zip",
#     root_dir="ensemble_pipeline"
# )

# print("Saved ensemble_pipeline1.zip")

import shutil
SAVE_DIR = "new_data_fineTuned_model"

zip_path = shutil.make_archive(
    base_dir=os.getcwd(),
    base_name = SAVE_DIR,
    format="zip",
    root_dir=SAVE_DIR
)


print("Saved files:", os.listdir(SAVE_DIR))
print("ZIP location:", zip_path)

Saved ensemble_pipeline1.zip


In [289]:
# import torch
# import shutil
# import os
# save_directory = "/kaggle/working/deberta_v3_finetuned"
# # DataParallel wrapper se original Hugging Face model nikalo
# model_to_save = (
#     model.module
#     if isinstance(model, torch.nn.DataParallel)
#     else model
# )
# # Save model weights + config
# model_to_save.save_pretrained(save_directory)

# # Save tokenizer files
# tokenizer.save_pretrained(save_directory)

# # Pack the entire directory into a ZIP
# zip_path = shutil.make_archive(
#     "/kaggle/working/deberta_v3_finetuned",  # ZIP ka output path aur naam
#     "zip",                                   #  Archive format ZIP   
#     root_dir=save_directory                  #  Kaunse folder ki files ZIP mein daalni hain
# )
# print("Saved files:", os.listdir(save_directory))
# print("ZIP location:", zip_path)

In [290]:
# for batch in train_loader:
#     # 1. Unpack your data (depending on what your Dataset __getitem__ returns)
#     input_ids = batch['input_ids']
#     attention_mask = batch['attention_mask']
#     labels = batch['labels']
#     print(input_ids.shape, attention_mask.shape, labels.shape)
#     print("*"*50)
#     print()
#     print
#     ()

# Print the total number of batches
# total_batches = len(train_loader)
# print(f"Total number of batches: {total_batches}")


In [291]:
# single_batch = next(iter(train_loader))

# # Check the shape of the tensor (e.g., torch.Size([batch_size, max_length]))
# print(single_batch['input_ids'].shape)

In [292]:
# batch = next(iter(train_loader))
# print("Input IDs", batch["input_ids"].shape)
# print("Attention mask", batch["attention_mask"].shape)
# print("Labels", batch["labels"].shape)

In [293]:
# outputs = model(
#     input_ids = batch["input_ids"],
#     attention_mask = batch["attention_mask"]
# )

# print(outputs.logits.shape)

In [294]:
# from torch.optim import AdamW
# optimizer = AdamW(
#     params=model.parameters(),
#     lr=2e-5  #Learning rate control karta hai ki optimizer parameters ko kitne scale par update kare.
# )

In [295]:
# from transformers import AutoTokenizer

# # Point directly to your local folder path
# LOCAL_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\checkpoint"

# # Load directly from the disk
# tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR)

# text = "Hello how is the microarchitecture of this overcompressive world"
# tokens = tokenizer.tokenize(text)
# print(tokens)


In [296]:
# print(len(tokenizer.tokenize(df.iloc[0]["text"])))

In [297]:
# text = "Ai is a cool tech."
# encoded = tokenizer(text)

# # Tokens 
# print("Tokens:")
# print(encoded.tokens())
# # Input ids
# print("Input Ids:")
# print(encoded["input_ids"])
# # attention mask (A 1 means: "This is a real word/token. Read it and process it.")
# print("Attention mask")
# print(encoded["attention_mask"])
# # tokens from ids
# print("Tokens from Ids:")
# print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))



In [298]:
# # 2. Complete tokenization
# encoded = tokenizer(text)

# print("\nInput IDs:")
# print(encoded["input_ids"])

# print("\nAttention Mask:")
# print(encoded["attention_mask"])

# # 3. IDs ko wapas tokens mein decode karo
# print("\nTokens from IDs:")
# print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

In [299]:
# import torch

# logits = torch.tensor([[2.0, 1.0, 0.1]])

# # Convert logits to probabilities
# probabilities = torch.softmax(logits, dim=1)

# # Find the class with the highest probability
# predicted_class = torch.argmax(probabilities, dim=1)

# print("Logits:", logits)
# print("Probabilities:", probabilities)
# print("Predicted class:", predicted_class.item())

In [300]:
# import torch
# print(torch.__version__)
# logits = torch.tensor([ 
#     1.2, -0.4, 2.8, 0.7,
#     0.2, 0.1, -0.8, 0.5,
#     1.0, -0.3, 0.4, 0.6,
#     0.0, 0.9, -0.2, 0.3,
#     0.8, 0.1])

# pred = torch.argmax(logits)
# print(pred.item())



In [301]:
# logits = torch.tensor([[2, 1, 0.1]])
# prob = torch.softmax(logits, dim=-1)
# print(torch.argmax(prob))
# # print(pred)

In [302]:
# probabilities = torch.softmax(logits, dim=-1)
# predictions = torch.argmax(logits, dim=-1)
# print(probabilities, predictions)

In [303]:
# import torch.nn as nn
# logits = torch.tensor([[2, 1, 0.1]])
# label = torch.tensor([1])
# loss_fn = nn.CrossEntropyLoss()
# loss = loss_fn(logits, label)
# print(loss.item())

In [304]:
# import os
# from transformers import AutoTokenizer

# MODEL_NAME = "microsoft/deberta-v3-small"
# HF_TOKEN = os.getenv("HF_TOKEN")

# # 1. Download from Hugging Face
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)

# # 2. Define a folder name on your disk
# LOCAL_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\checkpoint"

# # 3. Save it
# tokenizer.save_pretrained(LOCAL_DIR)
# print(f"Tokenizer successfully saved to {LOCAL_DIR}")


In [305]:
# from transformers import AutoTokenizer

# # Point directly to your local folder path
# LOCAL_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\checkpoint"

# # Load directly from the disk
# tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR)

# text = "Hello how is the microarchitecture of this overcompressive world"
# tokens = tokenizer.tokenize(text)
# print(tokens)
